In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load roads
roads = gpd.read_file(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\raw\roads\roads_102109.gpkg")

# Load study areas
study = gpd.read_file(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\study_areas.gpkg")

print("Roads loaded:", len(roads))
print("Study areas loaded:", len(study))
print("\nStudy area municipalities:")
print(study[['municipality', 'split']].to_string())

Roads loaded: 498431
Study areas loaded: 20

Study area municipalities:
          municipality  split
0           Ajdovščina  train
1                Koper  train
2                Kranj    val
3              Kočevje  train
4                Bovec  train
5               Tolmin    val
6                 Ptuj  train
7           Novo mesto    val
8        Murska Sobota  train
9                Piran   test
10            Postojna  train
11  Slovenska Bistrica  train
12               Celje  train
13           Ljubljana  train
14             Maribor  train
15             Domžale   test
16         Nova Gorica   test
17               Krško  train
18             Lendava   test
19              Bohinj   test


In [2]:
# Step 1 — Clip roads to study area extent
# First merge all study area polygons into one boundary
study_boundary = study.dissolve()

print("Clipping roads to study area boundary...")
roads_clipped = gpd.clip(roads, study_boundary)
print(f"Roads before clip: {len(roads):,}")
print(f"Roads after clip:  {len(roads_clipped):,}")
print(f"Reduction:         {100*(1 - len(roads_clipped)/len(roads)):.1f}%")

Clipping roads to study area boundary...
Roads before clip: 498,431
Roads after clip:  122,535
Reduction:         75.4%


In [3]:
# Step 2 — Assign road class and calculate buffer width

# Class mapping (GZ moved to Class 2 based on width analysis)
class_map = {
    'AC': 1, 'HC': 1, 'G1': 1, 'G2': 1, 'R1': 1,
    'GZ': 2, 'R2': 2, 'R3': 2, 'RT': 2, 'LC': 2, 'LZ': 2,
    'LG': 3, 'LK': 3, 'JP': 3, 'KD': 3, 'KJ': 3, 'NK': 3, 'PP': 3
}

# Minimum buffers per class (justified by 0.25m pixel resolution)
buffer_min = {1: 3.0, 2: 2.0, 3: 1.5}

# Fallback buffer when SIRCES=0
buffer_fallback = {1: 4.0, 2: 2.0, 3: 1.0}

# Work on a clean copy
roads_work = roads_clipped.copy()

# Drop the 2 rows with missing KAT_CES
roads_work = roads_work[roads_work['KAT_CES'].notna()].copy()

# Assign road class
roads_work['road_class'] = roads_work['KAT_CES'].map(class_map)

# Drop roads with no class mapping (should be 0 but let's verify)
unmapped = roads_work['road_class'].isna().sum()
print(f"Unmapped segments dropped: {unmapped}")
roads_work = roads_work[roads_work['road_class'].notna()].copy()

# Replace SIRCES=0 with class fallback
for cls in [1, 2, 3]:
    mask = (roads_work['road_class'] == cls) & (roads_work['SIRCES'] == 0)
    roads_work.loc[mask, 'SIRCES'] = buffer_fallback[cls] * 2
    print(f"Class {cls} SIRCES=0 replaced: {mask.sum()} segments")

# Calculate buffer = max(SIRCES/2, class_minimum)
roads_work['buffer_m'] = roads_work.apply(
    lambda row: max(row['SIRCES'] / 2, buffer_min[int(row['road_class'])]),
    axis=1
)

# Summary
print("\n=== Buffer width summary per class ===")
for cls in [1, 2, 3]:
    subset = roads_work[roads_work['road_class'] == cls]['buffer_m']
    print(f"\nClass {cls}:")
    print(f"  Count:  {len(subset):,}")
    print(f"  Mean:   {subset.mean():.2f}m")
    print(f"  Median: {subset.median():.2f}m")
    print(f"  Min:    {subset.min():.2f}m")
    print(f"  Max:    {subset.max():.2f}m")

Unmapped segments dropped: 0
Class 1 SIRCES=0 replaced: 0 segments
Class 2 SIRCES=0 replaced: 0 segments
Class 3 SIRCES=0 replaced: 0 segments

=== Buffer width summary per class ===

Class 1:
  Count:  2,523
  Mean:   4.20m
  Median: 4.00m
  Min:    3.00m
  Max:    15.00m

Class 2:
  Count:  8,136
  Mean:   2.86m
  Median: 2.50m
  Min:    2.00m
  Max:    18.50m

Class 3:
  Count:  111,874
  Mean:   1.85m
  Median: 1.50m
  Min:    1.50m
  Max:    19.50m


In [4]:
from shapely.geometry import CAP_STYLE, JOIN_STYLE
import warnings
warnings.filterwarnings('ignore')

print("Creating buffers... this may take 2-3 minutes")

# Apply buffer to each segment
roads_work['geometry'] = roads_work.apply(
    lambda row: row['geometry'].buffer(
        row['buffer_m'],
        cap_style=CAP_STYLE.flat,    # flat ends at road terminations
        join_style=JOIN_STYLE.mitre  # sharp corners at intersections
    ),
    axis=1
)

print("Buffers created.")

# Keep only essential columns
roads_buffered = roads_work[['road_class', 'KAT_CES', 'buffer_m', 'geometry']].copy()
roads_buffered['road_class'] = roads_buffered['road_class'].astype(int)

# Check result
print(f"\nTotal buffered polygons: {len(roads_buffered):,}")
print(f"\nGeometry types:")
print(roads_buffered.geometry.geom_type.value_counts())

# Quick area check per class
print("\n=== Total buffered area per class ===")
for cls in [1, 2, 3]:
    subset = roads_buffered[roads_buffered['road_class'] == cls]
    total_area = subset.geometry.area.sum() / 1_000_000  # km²
    print(f"Class {cls}: {total_area:.2f} km²")

Creating buffers... this may take 2-3 minutes
Buffers created.

Total buffered polygons: 122,533

Geometry types:
Polygon         121894
MultiPolygon       639
Name: count, dtype: int64

=== Total buffered area per class ===
Class 1: 8.26 km²
Class 2: 20.93 km²
Class 3: 71.55 km²


In [5]:
print("Handling class priority overlaps...")
print("Rule: Class 1 > Class 2 > Class 3")

# Separate by class
class1 = roads_buffered[roads_buffered['road_class'] == 1].copy()
class2 = roads_buffered[roads_buffered['road_class'] == 2].copy()
class3 = roads_buffered[roads_buffered['road_class'] == 3].copy()

# Dissolve each class into a single geometry
print("Dissolving Class 1...")
geom1 = class1.dissolve()['geometry'].iloc[0]

print("Dissolving Class 2...")
geom2 = class2.dissolve()['geometry'].iloc[0]

print("Dissolving Class 3...")
geom3 = class3.dissolve()['geometry'].iloc[0]

# Remove higher class areas from lower classes
print("Applying priority rules...")
geom2_clean = geom2.difference(geom1)          # Class 2 minus Class 1
geom3_clean = geom3.difference(geom1).difference(geom2)  # Class 3 minus Class 1 and 2

# Build final GeoDataFrame
from shapely.geometry import MultiPolygon
import geopandas as gpd

final_roads = gpd.GeoDataFrame(
    {
        'road_class': [1, 2, 3],
        'class_name': ['major_roads', 'local_roads', 'minor_roads'],
        'geometry': [geom1, geom2_clean, geom3_clean]
    },
    crs=roads_buffered.crs
)

# Area check after priority handling
print("\n=== Area after priority handling ===")
for idx, row in final_roads.iterrows():
    area_km2 = row['geometry'].area / 1_000_000
    print(f"Class {row['road_class']} ({row['class_name']}): {area_km2:.2f} km²")

print("\nDone.")

Handling class priority overlaps...
Rule: Class 1 > Class 2 > Class 3
Dissolving Class 1...
Dissolving Class 2...
Dissolving Class 3...
Applying priority rules...

=== Area after priority handling ===
Class 1 (major_roads): 8.09 km²
Class 2 (local_roads): 20.75 km²
Class 3 (minor_roads): 69.92 km²

Done.


In [6]:
import os

# Save buffered road polygons
output_path = r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\road_labels\road_labels_buffered.gpkg"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print("Saving buffered road labels...")
final_roads.to_file(output_path, driver='GPKG')
print(f"Saved to: {output_path}")

# Verify by reloading
verify = gpd.read_file(output_path)
print(f"\nVerification - rows loaded back: {len(verify)}")
print(verify[['road_class', 'class_name']].to_string())

Saving buffered road labels...
Saved to: C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\road_labels\road_labels_buffered.gpkg

Verification - rows loaded back: 3
   road_class   class_name
0           1  major_roads
1           2  local_roads
2           3  minor_roads


In [1]:
import time
import numpy as np
from pathlib import Path

test_path = Path(r"\\kgkn-nas\eo_data_2\GURS_podatki\DOF\DOF025_tiles\speed_test.npy")
data = np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)

t = time.time()
for i in range(10):
    np.save(test_path, data)
elapsed = time.time() - t

print(f"10 files written in {elapsed:.2f}s")
print(f"Speed: {10/elapsed:.1f} files/second")
print(f"Estimated time for 470,000 tiles: {470000/(10/elapsed)/3600:.1f} hours")

10 files written in 0.26s
Speed: 38.0 files/second
Estimated time for 470,000 tiles: 3.4 hours
